# Video Subtitle Extractor — Google Colab (GPU)

Extract hard-coded subtitles from videos using GPU acceleration.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Upload your video file (left sidebar → 📁 → upload)
3. Run all cells below
4. Download the `.srt` subtitle file

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install Python 3.12 (Colab default is 3.10/3.11)
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3.12 python3.12-venv python3.12-dev
!python3.12 -m ensurepip --upgrade

In [ ]:
# Clone the repo
!git clone https://github.com/YaoFANGUK/video-subtitle-extractor.git
%cd video-subtitle-extractor

In [ ]:
# Detect CUDA version and install PaddlePaddle GPU
import subprocess, sys

cuda_version = None
try:
    nvcc = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    for line in nvcc.stdout.split('\n'):
        if 'release' in line:
            cuda_version = line.split('release')[1].strip().split(',')[0].strip()
            break
except:
    pass

print(f'Detected CUDA: {cuda_version}')

# Map CUDA version to paddlepaddle index
if cuda_version and cuda_version.startswith('12.'):
    minor = cuda_version.split('.')[1]
    if int(minor) >= 8:
        index = 'https://www.paddlepaddle.org.cn/packages/stable/cu128/'
    elif int(minor) >= 6:
        index = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
    else:
        index = 'https://www.paddlepaddle.org.cn/packages/stable/cu120/'
else:
    index = 'https://www.paddlepaddle.org.cn/packages/stable/cu118/'

print(f'Using index: {index}')

# Install PaddlePaddle GPU + dependencies (skip paddlepaddle in requirements)
!python3.12 -m pip install paddlepaddle-gpu==3.2.0 -i $index
!python3.12 -m pip install -r requirements.txt
!python3.12 -m pip install google-colab

In [ ]:
# Set video path — upload your video to Colab first, then set path here
from google.colab import files
import os

uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f'Video: {video_path}')
print(f'Size: {os.path.getsize(video_path) / 1024 / 1024:.1f} MB')

### Configure subtitle area (optional)

Set coordinates for where subtitles appear. Leave as defaults for bottom subtitles (80-100% height).

Format: `ymin ymax xmin xmax` (pixel coordinates)
- For 1920×1080 video with bottom subtitles: leave default
- For 1280×720 video: use `560 710 50 1230`
- Leave empty for full-frame detection (slower)

In [ ]:
# Get video dimensions to calculate subtitle area
import cv2
cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f'Resolution: {width}×{height}')
print(f'FPS: {fps:.1f}')
print(f'Frames: {frames}')
print(f'Duration: {frames/fps:.1f}s')

# Default: bottom 20% of video (typical subtitle placement)
ymin = int(height * 0.78)
ymax = int(height * 0.99)
xmin = int(width * 0.05)
xmax = int(width * 0.95)

print(f'\nSubtitle area (auto-calculated):')
print(f'  ymin={ymin}, ymax={ymax}, xmin={xmin}, xmax={xmax}')
print(f'  Region: {xmax-xmin}×{ymax-ymin} px')

In [ ]:
# Run subtitle extraction
import sys, os, multiprocessing
sys.path.insert(0, '.')

# Ensure GPU is enabled
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

multiprocessing.set_start_method('spawn')

from backend.main import SubtitleExtractor
from backend.bean.subtitle_area import SubtitleArea
from backend.config import config

# Configure
config.set(config.language, 'ch')
config.set(config.mode, 'fast')

sub_area = SubtitleArea(ymin, ymax, xmin, xmax)

se = SubtitleExtractor(video_path)
se.sub_area = sub_area
se.subtitle_output_path = video_path.rsplit('.', 1)[0] + '.srt'
se.run()

print('\nDone!')

In [ ]:
# Download the subtitle file
srt_path = video_path.rsplit('.', 1)[0] + '.srt'

if os.path.exists(srt_path) and os.path.getsize(srt_path) > 0:
    from google.colab import files
    files.download(srt_path)
    print(f'Downloaded: {srt_path}')
else:
    print('ERROR: SRT file not generated or empty. Check logs above.')